# MHQA — RETRIEVER FINE-TUNE (contrastive) -> COPY pipeline   [Lug_Uga]
#
─────────────────────────────────────────────────────────────────────────────
 WHAT THIS DOES (evidence: full-data audit 2026-06-10)
   Lug_Uga: 87.6% of val answers exist VERBATIM in the train bank, but 0%
   of questions repeat -> pure PARAPHRASE-RETRIEVAL problem. Oracle ceiling
   = 0.90 combined. Zero-shot e5_large copy = 0.50 -> the gap is recall@1.
   Fix: fine-tune the retriever on the dataset's own supervision:
     * 830 same-answer clusters covering 91% of rows -> 10,804 positive
       question pairs (questions sharing an identical answer = paraphrases)
     * hard negatives mined from the base retriever's wrong neighbors
     * MultipleNegativesRankingLoss (in-batch negs + 1 mined hard neg)
 PIPELINE
   1. Load Train.csv + Val.csv, clean, COMBINE, select SELECT_SUBSETS.
   2. StratifiedKFold on ANSWER-LENGTH bins (5 bins, seed 42) — as requested.
   3. Per fold: build pairs from TRAIN FOLD ONLY (no val leakage), mine hard
      negatives INSIDE the train fold, fine-tune, then OOF copy eval on the
      val fold (bank = train-fold questions). Reports BASE vs FINE-TUNED.
   4. Final: retrain on ALL rows' pairs, save encoder, copy answers for
      Test.csv (bank = all selected rows), emit 4-col submission.
 METRIC: unicode ROUGE (NFKC + lowercase + [^\w] strip) = corrected-LB proxy.
 FALLBACK: if sentence-transformers/torch unavailable, fine-tuning is skipped
   and a char-TFIDF retriever runs the same plumbing (sandbox-verifiable).
─────────────────────────────────────────────────────────────────────────────

In [16]:
import os, sys, subprocess
WHEELS_DIR = '/kaggle/input/datasets/haniagamal/teleco-wheels/wheels'   # '' to skip
def _offline_pip(pkgs):
    if not WHEELS_DIR: print(f'Skipping pip; need pre-installed: {pkgs}'); return
    try:
        subprocess.run([sys.executable,'-m','pip','install','-q','--no-index',
                        f'--find-links={WHEELS_DIR}',*pkgs],check=True)
        print(f'Installed from wheels: {pkgs}')
    except Exception as e: print(f'Offline pip failed for {pkgs}: {e}')
_offline_pip(['rouge-score'])
os.environ['HF_HUB_OFFLINE']='1'; os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['HF_DATASETS_OFFLINE']='1'; os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
import re, gc, random, unicodedata, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('Imports complete')

Installed from wheels: ['rouge-score']
Imports complete


In [17]:
MODEL_KEY = 'afrie5'            # 'afrie5' | 'e5_large' | 'bge_m3' | 'labse'
                                # afrie5 first for Luganda; run e5_large second, keep better OOF.
RETRIEVER_REGISTRY = {
    'afrie5':   {'path': '/kaggle/input/datasets/haniagamal/mhqa-models-data/models/afrie5',
                 'q_prefix': '',         'd_prefix': ''},
    'e5_large': {'path': '/kaggle/input/datasets/haniagamal/mhqa-models-data/models/e5_large',
                 'q_prefix': 'query: ',  'd_prefix': 'passage: '},
    'bge_m3':   {'path': '/kaggle/input/datasets/haniagamal/mhqa-models-data/models/bge_m3',
                 'q_prefix': '',         'd_prefix': ''},
    'labse':    {'path': '/kaggle/input/datasets/haniagamal/mhqa-models-data/models/labse',
                 'q_prefix': '',         'd_prefix': ''},
}
RET_CFG  = RETRIEVER_REGISTRY[MODEL_KEY]
SELECT_SUBSETS = ['Lug_Uga']
# folds — StratifiedKFold on ANSWER-LENGTH bins (as requested)
N_SPLITS      = 5
N_LEN_BINS    = 5
FOLDS_TO_RUN  = [0, 1, 2, 3, 4]      # all 5 for a real OOF; [0] for a quick smoke run
# pair building
PAIRS_PER_CLUSTER   = 15             # cap (largest cluster = 30 members -> 435 raw pairs)
MAX_TRAIN_PAIRS     = 25000
USE_HARD_NEGATIVES  = True
HARD_NEGS_PER_PAIR  = 1              # MNRL triplet (anchor, positive, hard_neg)
HARD_NEG_TOPK       = 8              # mine negs from base-retriever top-K wrong-cluster hits
# fine-tuning
FT_EPOCHS     = 2
FT_BATCH      = 64
FT_LR         = 2e-5
FT_WARMUP_FRAC= 0.10
FT_MAX_SEQ    = 128
EVAL_BASE_TOO = True                 # also score the BASE model per fold (the delta is the proof)
# output
WORK_DIR  = Path('/kaggle/working')
FINAL_MODEL_DIR = WORK_DIR / f'retriever_ft_{MODEL_KEY}'
_sub = ('_'+'_'.join(SELECT_SUBSETS)) if SELECT_SUBSETS else ''
OUTPUT_SUBMISSION = WORK_DIR / f'submission_retft_{MODEL_KEY}{_sub}.csv'
OOF_CSV           = WORK_DIR / f'oof_retft_{MODEL_KEY}{_sub}.csv'
print(f'Retriever: {MODEL_KEY} ({RET_CFG["path"]})')
print(f'Subsets={SELECT_SUBSETS} | folds={FOLDS_TO_RUN}/{N_SPLITS} (answer-length bins={N_LEN_BINS})')
print(f'Pairs: cap/cluster={PAIRS_PER_CLUSTER} max={MAX_TRAIN_PAIRS} | hard-negs={USE_HARD_NEGATIVES}')
print(f'FT: epochs={FT_EPOCHS} batch={FT_BATCH} lr={FT_LR}')

Retriever: afrie5 (/kaggle/input/datasets/haniagamal/mhqa-models-data/models/afrie5)
Subsets=['Lug_Uga'] | folds=[0, 1, 2, 3, 4]/5 (answer-length bins=5)
Pairs: cap/cluster=15 max=25000 | hard-negs=True
FT: epochs=2 batch=64 lr=2e-05


In [18]:
DATA_DIR = Path('/kaggle/input/datasets/haniagamal/repo-mhqa/multilingual-health-qa/data')
ID_COL, QUESTION_COL, ANSWER_COL, LANG_COL = 'ID', 'input', 'output', 'subset'
def clean_text(x): return '' if pd.isna(x) else str(x).strip()
train = pd.read_csv(DATA_DIR/'Train.csv'); val = pd.read_csv(DATA_DIR/'Val.csv')
test  = pd.read_csv(DATA_DIR/'Test.csv')
for df in (train, val):
    df[QUESTION_COL] = df[QUESTION_COL].map(clean_text)
    df[ANSWER_COL]   = df[ANSWER_COL].map(clean_text)
test[QUESTION_COL] = test[QUESTION_COL].map(clean_text)
print(f'Raw Train {train.shape} | Val {val.shape} | Test {test.shape}')
full = pd.concat([train, val], ignore_index=True)              # COMBINED, as requested
full = full[(full[QUESTION_COL]!='') & (full[ANSWER_COL]!='')]
full = full.drop_duplicates(subset=[QUESTION_COL, ANSWER_COL, LANG_COL]).reset_index(drop=True)
if SELECT_SUBSETS:
    full = full[full[LANG_COL].isin(SELECT_SUBSETS)].reset_index(drop=True)
    if LANG_COL in test.columns:
        _n0 = len(test); test = test[test[LANG_COL].isin(SELECT_SUBSETS)].reset_index(drop=True)
        print(f'Test restricted to {SELECT_SUBSETS}: {len(test)}/{_n0} rows')
print(f'Combined+selected: {len(full)} rows | unique answers: {full[ANSWER_COL].nunique()}')

Raw Train (29815, 4) | Val (6686, 4) | Test (2618, 3)
Test restricted to ['Lug_Uga']: 374/2618 rows
Combined+selected: 4229 rows | unique answers: 1205


In [19]:
_lens = full[ANSWER_COL].astype(str).str.split().apply(len)
try:
    LEN_BINS = pd.qcut(_lens, q=N_LEN_BINS, labels=False, duplicates='drop').fillna(0).astype(int).values
except Exception:
    LEN_BINS = pd.cut(_lens, bins=N_LEN_BINS, labels=False).fillna(0).astype(int).values
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
print(f'Fold scheme: StratifiedKFold[answer-length, {pd.Series(LEN_BINS).nunique()} bins] '
      f'(bin sizes: {pd.Series(LEN_BINS).value_counts().sort_index().tolist()})')

Fold scheme: StratifiedKFold[answer-length, 5 bins] (bin sizes: [865, 839, 860, 826, 839])


In [20]:
def _utok(s):
    s = unicodedata.normalize('NFKC', str(s)).lower()
    return [x for x in re.sub(r'[^\w]+', ' ', s, flags=re.UNICODE).split() if x]
def _lcs(a, b):
    if not a or not b: return 0
    p = [0]*(len(b)+1)
    for i in range(1, len(a)+1):
        c = [0]*(len(b)+1); ai = a[i-1]
        for j in range(1, len(b)+1):
            c[j] = p[j-1]+1 if ai == b[j-1] else max(p[j], c[j-1])
        p = c
    return p[-1]
def _f1(o, n, r): return 0.0 if (o==0 or n==0 or r==0) else 2*(o/n)*(o/r)/((o/n)+(o/r))
def pair_scores(ref, pred):
    rt, pt = _utok(ref), _utok(pred)
    r1 = _f1(sum((Counter(rt)&Counter(pt)).values()), len(pt), len(rt))
    rl = _f1(_lcs(rt, pt), len(pt), len(rt))
    return r1, rl
def combined(preds, refs):
    s = [pair_scores(r, p) for p, r in zip(preds, refs)]
    r1 = float(np.mean([x[0] for x in s])); rl = float(np.mean([x[1] for x in s]))
    return {'R1': r1, 'RL': rl, 'COMB': 0.5*r1+0.5*rl}
print('Metric ready: unicode ROUGE, COMBINED = 0.5*R1 + 0.5*RL')

Metric ready: unicode ROUGE, COMBINED = 0.5*R1 + 0.5*RL


In [21]:
HAVE_ST = False
try:
    import torch
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader
    HAVE_ST = True
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'sentence-transformers OK | device={DEVICE}')
except Exception as e:
    DEVICE = 'cpu'
    print(f'[fallback] sentence-transformers/torch unavailable ({type(e).__name__}) -> '
          f'char-TFIDF retriever, FINE-TUNING SKIPPED (plumbing-verification mode)')

class DenseRetriever:
    def __init__(self, model): self.m = model
    def _enc(self, texts, prefix):
        return np.asarray(self.m.encode([prefix+str(t) for t in texts], batch_size=128,
                                        normalize_embeddings=True, show_progress_bar=False),
                          dtype='float32')
    def fit(self, bank_q, d_prefix): self.D = self._enc(bank_q, d_prefix)
    def sims(self, query_q, q_prefix): return self._enc(query_q, q_prefix) @ self.D.T

class TfidfRetriever:
    def fit(self, bank_q, d_prefix=''):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.v = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2)
        self.D = self.v.fit_transform([str(x) for x in bank_q])
    def sims(self, query_q, q_prefix=''):
        return np.asarray((self.v.transform([str(x) for x in query_q]) @ self.D.T).todense())

def load_base_st():
    return SentenceTransformer(RET_CFG['path'], device=DEVICE)

sentence-transformers OK | device=cuda


In [22]:
def build_pairs(df_bank, hard_neg_retriever=None, verbose=True):
    """positives = question pairs sharing an IDENTICAL answer, built ONLY from df_bank.
       hard negatives = base-retriever top-K neighbors from a DIFFERENT cluster."""
    qs = df_bank[QUESTION_COL].values
    ans = df_bank[ANSWER_COL].values
    a2rows = defaultdict(list)
    for i, a in enumerate(ans): a2rows[a].append(i)
    clusters = [v for v in a2rows.values() if len(v) >= 2]
    rng = np.random.RandomState(SEED)
    raw_pairs = []
    for rows in clusters:
        cand = [(rows[i], rows[j]) for i in range(len(rows)) for j in range(i+1, len(rows))]
        if len(cand) > PAIRS_PER_CLUSTER:
            cand = [cand[k] for k in rng.choice(len(cand), PAIRS_PER_CLUSTER, replace=False)]
        raw_pairs.extend(cand)
    if len(raw_pairs) > MAX_TRAIN_PAIRS:
        raw_pairs = [raw_pairs[k] for k in rng.choice(len(raw_pairs), MAX_TRAIN_PAIRS, replace=False)]
    # hard negatives: for each anchor, nearest bank questions NOT in the anchor's cluster
    hard_neg_of = {}
    if USE_HARD_NEGATIVES and hard_neg_retriever is not None and len(raw_pairs):
        anchors = sorted({a for a, _ in raw_pairs})
        S = hard_neg_retriever.sims([qs[a] for a in anchors], RET_CFG['q_prefix'])
        order = np.argsort(-S, axis=1)[:, :HARD_NEG_TOPK+4]
        for ai, a in enumerate(anchors):
            own = set(a2rows[ans[a]])
            negs = [j for j in order[ai] if j not in own and j != a][:HARD_NEGS_PER_PAIR]
            if negs: hard_neg_of[a] = negs
    n_trip = sum(1 for a, _ in raw_pairs if a in hard_neg_of)
    if verbose:
        print(f'  pairs: {len(raw_pairs):,} from {len(clusters)} clusters '
              f'(rows in clusters: {sum(len(v) for v in clusters)}/{len(df_bank)}) '
              f'| with hard-neg: {n_trip:,}')
    return qs, raw_pairs, hard_neg_of

def make_examples(qs, raw_pairs, hard_neg_of):
    """symmetric pairs both directions; e5 prefixes: anchor=query:, pos/neg=passage:"""
    qp, dp = RET_CFG['q_prefix'], RET_CFG['d_prefix']
    ex = []
    for a, b in raw_pairs:
        for x, y in ((a, b), (b, a)):
            t = [qp+str(qs[x]), dp+str(qs[y])]
            if x in hard_neg_of: t.append(dp+str(qs[hard_neg_of[x][0]]))
            ex.append(InputExample(texts=t))
    return ex

def finetune_on(df_bank, tag):
    """returns a retriever (fine-tuned dense if available, else tfidf fallback)."""
    if not HAVE_ST:
        r = TfidfRetriever(); 
        print(f'  [{tag}] FT skipped (no sentence-transformers) -> TFIDF retriever')
        _ = build_pairs(df_bank, hard_neg_retriever=_tfidf_for_mining(df_bank))   # still verify pair counts
        return r
    base = load_base_st(); base.max_seq_length = FT_MAX_SEQ
    miner = DenseRetriever(base); miner.fit(df_bank[QUESTION_COL].values, RET_CFG['d_prefix'])
    qs, raw_pairs, hn = build_pairs(df_bank, hard_neg_retriever=miner)
    if not raw_pairs:
        print(f'  [{tag}] no pairs -> returning base model'); return DenseRetriever(base)
    examples = make_examples(qs, raw_pairs, hn)
    loader = DataLoader(examples, shuffle=True, batch_size=FT_BATCH, drop_last=True)
    loss = losses.MultipleNegativesRankingLoss(base)
    steps = max(1, len(loader)) * FT_EPOCHS
    base.fit(train_objectives=[(loader, loss)], epochs=FT_EPOCHS,
             warmup_steps=int(FT_WARMUP_FRAC*steps), optimizer_params={'lr': FT_LR},
             show_progress_bar=True)
    print(f'  [{tag}] fine-tuned on {len(examples):,} examples ({FT_EPOCHS} epochs)')
    return DenseRetriever(base)

def _tfidf_for_mining(df_bank):
    r = TfidfRetriever(); r.fit(df_bank[QUESTION_COL].values); return r

In [23]:
def copy_with(retriever, bank_df, query_qs):
    retriever.fit(bank_df[QUESTION_COL].values, RET_CFG['d_prefix'])
    S = retriever.sims(query_qs, RET_CFG['q_prefix'])
    top1 = np.argmax(S, axis=1)
    return bank_df[ANSWER_COL].values[top1], S[np.arange(len(query_qs)), top1]

OOF_PRED = np.empty(len(full), object); OOF_SIM = np.full(len(full), np.nan)
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(splitter.split(full, LEN_BINS)):
    if fold not in FOLDS_TO_RUN: continue
    print('\n' + '='*74 + f'\nFOLD {fold} | bank={len(tr_idx):,} val={len(va_idx):,} | retriever={MODEL_KEY}\n' + '='*74)
    bank_df = full.iloc[tr_idx].reset_index(drop=True)
    vq   = full[QUESTION_COL].values[va_idx]
    refs = full[ANSWER_COL].values[va_idx]
    row = {'fold': fold}
    if EVAL_BASE_TOO and HAVE_ST:
        base_r = DenseRetriever(load_base_st())
        p0, _ = copy_with(base_r, bank_df, vq)
        m0 = combined(list(p0), list(refs)); row.update({'base_COMB': round(m0['COMB'],4)})
        print(f'  BASE copy   COMB={m0["COMB"]:.4f} (R1 {m0["R1"]:.4f}/RL {m0["RL"]:.4f})')
        del base_r; gc.collect()
        if HAVE_ST and DEVICE=='cuda': torch.cuda.empty_cache()
    ft = finetune_on(bank_df, tag=f'fold{fold}')
    preds, sims = copy_with(ft, bank_df, vq)
    m = combined(list(preds), list(refs))
    row.update({'ft_COMB': round(m['COMB'],4), 'ft_R1': round(m['R1'],4), 'ft_RL': round(m['RL'],4),
                'sim_mean': round(float(np.mean(sims)),3)})
    print(f'  FT    copy   COMB={m["COMB"]:.4f} (R1 {m["R1"]:.4f}/RL {m["RL"]:.4f}) | top-1 sim mean={np.mean(sims):.3f}')
    OOF_PRED[va_idx] = np.array(preds, dtype=object); OOF_SIM[va_idx] = sims
    fold_rows.append(row)
    del ft; gc.collect()
    if HAVE_ST and DEVICE=='cuda': torch.cuda.empty_cache()

print('\n' + '='*74 + '\nPER-FOLD SUMMARY\n' + '='*74)
fold_tbl = pd.DataFrame(fold_rows); print(fold_tbl.to_string(index=False))
mask = np.array([p is not None for p in OOF_PRED])
if mask.any():
    oof_m = combined(list(OOF_PRED[mask]), list(full[ANSWER_COL].values[mask]))
    print(f'\nOOF over {int(mask.sum())} rows: COMB={oof_m["COMB"]:.4f} (R1 {oof_m["R1"]:.4f}/RL {oof_m["RL"]:.4f})')
    pd.DataFrame({'ID': full[ID_COL].values[mask], 'subset': full[LANG_COL].values[mask],
                  'sim_top1': OOF_SIM[mask], 'ref': full[ANSWER_COL].values[mask],
                  'pred': OOF_PRED[mask]}).to_csv(OOF_CSV, index=False)
    print(f'[saved] {OOF_CSV}')


FOLD 0 | bank=3,383 val=846 | retriever=afrie5


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  BASE copy   COMB=0.5408 (R1 0.5605/RL 0.5211)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  pairs: 4,482 from 726 clusters (rows in clusters: 2982/3383) | with hard-neg: 4,482


...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  pairs: 4,500 from 722 clusters (rows in clusters: 2995/3384) | with hard-neg: 4,500


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


  [fold4] fine-tuned on 9,000 examples (2 epochs)
  FT    copy   COMB=0.6607 (R1 0.6762/RL 0.6452) | top-1 sim mean=0.839

PER-FOLD SUMMARY
 fold  base_COMB  ft_COMB  ft_R1  ft_RL  sim_mean
    0     0.5408   0.6514 0.6673 0.6355     0.845
    1     0.5495   0.6663 0.6819 0.6506     0.845
    2     0.5255   0.6516 0.6669 0.6363     0.848
    3     0.5474   0.6624 0.6775 0.6474     0.849
    4     0.5254   0.6607 0.6762 0.6452     0.839

OOF over 4229 rows: COMB=0.6585 (R1 0.6739/RL 0.6430)
[saved] /kaggle/working/oof_retft_afrie5_Lug_Uga.csv


In [ ]:
print('\n' + '='*74 + '\nFINAL MODEL (all rows) + TEST PREDICTION\n' + '='*74)
final_ret = finetune_on(full, tag='final-all-data')
if HAVE_ST:
    final_ret.m.save(str(FINAL_MODEL_DIR)); print(f'[saved] fine-tuned encoder -> {FINAL_MODEL_DIR}')
test_preds, test_sims = copy_with(final_ret, full, test[QUESTION_COL].values)
print(f'test top-1 sim: mean={np.mean(test_sims):.3f} median={np.median(test_sims):.3f}')
out = [str(p).strip() or (str(q).strip()[:300] or 'N/A')
       for p, q in zip(test_preds, test[QUESTION_COL].values)]
sub = pd.DataFrame({'ID': test[ID_COL].values, 'TargetRLF1': out, 'TargetR1F1': out,
                    'TargetLLM': out})[['ID','TargetRLF1','TargetR1F1','TargetLLM']]
assert len(sub) == len(test)
sub.to_csv(OUTPUT_SUBMISSION, index=False, encoding='utf-8')
print(f'Saved {OUTPUT_SUBMISSION}  shape={sub.shape}')
print(sub.head(3).to_string(index=False))
print('\nNEXT: run MODEL_KEY="e5_large" too and keep the better OOF; '
      'the saved encoder also drops into the F17 copy harness via LOCAL_MODEL_PATHS.')


FINAL MODEL (all rows) + TEST PREDICTION


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  pairs: 5,965 from 830 clusters (rows in clusters: 3854/4229) | with hard-neg: 5,964


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
